In [ ]:
# ==========================================================
# Cell 3 : Read Evaluation Dataset (Top 50 XML Sources)
# ==========================================================

from pyspark.sql import functions as F

# ----------------------------------------------------------
# Configuration
# ----------------------------------------------------------

RAG_EVAL_TABLE = "knwbt_airr_foundry_dev_catalog.secured.rag_eval_set"

TOP_K_XML = 50

# ----------------------------------------------------------
# Read Evaluation Dataset
# ----------------------------------------------------------

rag_eval_df = (

    spark.table(RAG_EVAL_TABLE)

)

print("=" * 80)
print("RAG EVALUATION DATASET")
print("=" * 80)

print(f"Total Questions : {rag_eval_df.count()}")

# ----------------------------------------------------------
# Select Required Columns
# ----------------------------------------------------------

rag_eval_df = (

    rag_eval_df

    .select(

        "doc_id",
        "request",
        "question_type",
        "difficulty",
        "ground_truth",
        "source_doc_ref",
        "source_doc_title",
        "source",
        "expected_facts",
        "retrieved_context"

    )

)

# ----------------------------------------------------------
# Take Top 50 XML Sources
# ----------------------------------------------------------

top50_eval = (

    rag_eval_df

    .limit(TOP_K_XML)

)

# ----------------------------------------------------------
# Remove Duplicate XML Paths
# ----------------------------------------------------------

top50_sources = (

    top50_eval

    .select("source")

    .dropDuplicates()

)

print()

print("=" * 80)
print("TOP XML SOURCES")
print("=" * 80)

print(f"Questions Selected : {top50_eval.count()}")

print(f"Unique XML Sources : {top50_sources.count()}")

print()

display(top50_eval)

print()

display(top50_sources)






In [ ]:


# ==========================================================
# Cell 4 : Extract Matching XML Documents
# ==========================================================

from pyspark.sql import functions as F

# ----------------------------------------------------------
# Read Raw XML Table
# ----------------------------------------------------------

RAW_TABLE = "knwbt_airr_foundry_dev_catalog.secured.knowbot_raw_data"

raw_df = spark.table(RAW_TABLE)

print("=" * 80)
print("KNOWBOT RAW DATA")
print("=" * 80)

print(f"Total XML Documents : {raw_df.count()}")

# ----------------------------------------------------------
# Match XML Paths
# ----------------------------------------------------------

matched_xml = (

    raw_df.alias("raw")

    .join(

        top50_sources.alias("eval"),

        F.col("raw.source") == F.col("eval.source"),

        "inner"

    )

)

# ----------------------------------------------------------
# Remove Duplicate XML Documents
# ----------------------------------------------------------

matched_xml = (

    matched_xml

    .dropDuplicates(["doc_id"])

)

# ----------------------------------------------------------
# Select Required Columns
# ----------------------------------------------------------

matched_xml = (

    matched_xml

    .select(

        "doc_id",

        "title",

        "content",

        "reference",

        "source",

        "version",

        "content_update_date",

        "content_author"

    )

)

print()

print("=" * 80)
print("MATCHED XML DOCUMENTS")
print("=" * 80)

print(f"Matched XML Documents : {matched_xml.count()}")

print()

display(matched_xml)


#################################  5  ##########################
# ==========================================================
# Cell 5 : XML Data Cleaning
# ==========================================================

from pyspark.sql import functions as F

print("=" * 80)
print("XML DATA CLEANING")
print("=" * 80)

# ----------------------------------------------------------
# Input from Cell 4
# ----------------------------------------------------------

xml_df = matched_xml

print(f"Matched XML Documents : {xml_df.count()}")

# ----------------------------------------------------------
# Remove Duplicate XML Documents
# ----------------------------------------------------------

xml_df = (

    xml_df

    .dropDuplicates(

        ["doc_id"]

    )

)

# ----------------------------------------------------------
# Replace NULL Content
# (Do NOT remove XML documents)
# ----------------------------------------------------------

xml_df = (

    xml_df

    .fillna(

        {

            "content": ""

        }

    )

)

# ----------------------------------------------------------
# Clean XML Content
# ----------------------------------------------------------

xml_df = (

    xml_df

    .withColumn(

        "content",

        F.regexp_replace(

            F.col("content"),

            r"\s+",

            " "

        )

    )

)

# ----------------------------------------------------------
# Trim Text Columns
# ----------------------------------------------------------

xml_df = (

    xml_df

    .withColumn(

        "title",

        F.trim(

            F.col("title")

        )

    )

    .withColumn(

        "reference",

        F.trim(

            F.col("reference")

        )

    )

    .withColumn(

        "source",

        F.trim(

            F.col("source")

        )

    )

)

# ----------------------------------------------------------
# Summary
# ----------------------------------------------------------

print()

print("=" * 80)
print("CLEAN XML DATA")
print("=" * 80)

print(f"XML Documents Ready : {xml_df.count()}")

print()

display(xml_df)

In [ ]:
import re
from pyspark.sql import Row

# ==========================================================
# Cell 4 : Clean XML Content
# ==========================================================

def clean_text(text):
    """
    Clean XML content while preserving GraphRAG links.

    Preserves:
        [[doc:...]]
        [[doc:wiki:...]]
        [[doc:Temp_Import...]]
    """

    if not text:
        return ""

    # ------------------------------------------------------
    # Remove CDATA markers only
    # ------------------------------------------------------

    text = re.sub(r"<!\[CDATA\[|\]\]>", "", text)

    # ------------------------------------------------------
    # Normalize whitespace
    # (Does NOT modify document references)
    # ------------------------------------------------------

    text = re.sub(r"\s+", " ", text)

    return text.strip()


# ==========================================================
# Create Clean DataFrame
# ==========================================================

clean_rows = []

for row in df_raw.collect():

    clean_rows.append(

        Row(

            doc_id=row["doc_id"],

            source_path=row["source_path"],

            title=(row["title"] or "").strip(),

            reference=(row["reference"] or "").strip(),

            version=(row["version"] or "").strip(),

            content_update_date=(row["content_update_date"] or "").strip(),

            content_author=(row["content_author"] or "").strip(),

            # Preserve doc/doc:wiki/doc:Temp_Import references
            content=clean_text(row["content"])

        )

    )

# ==========================================================
# Spark DataFrame
# ==========================================================

df_clean = spark.createDataFrame(clean_rows)

# ==========================================================
# Summary
# ==========================================================

print("=" * 80)
print("XML CLEANING SUMMARY")
print("=" * 80)
print(f"Total Clean Documents : {df_clean.count()}")
print("=" * 80)

display(df_clean)

cell 5 will be the clean part XML data cleaning

cell 6 will be the Build reference resolution Engine i will provied the image of pic of this code of cell6 

In [ ]:
# ==========================================================
# Cell 7 : Universal XML Reference Parser
# ==========================================================

import re

# ==========================================================
# Normalize Reference
# ==========================================================

def normalize_reference(ref):

    if not ref:
        return None

    ref = ref.strip()

    # Remove anchor
    if "||" in ref:
        ref = ref.split("||")[0]

    # Remove section
    if "#" in ref:
        ref = ref.split("#")[0]

    # Decode HTML entities
    ref = ref.replace("&amp;", "&")

    # Normalize spaces
    ref = re.sub(r"\s+", " ", ref)

    return ref.strip()


# ==========================================================
# Universal Relation Parser
# ==========================================================

def parse_link(raw):

    raw = raw.strip()

    # Only process document links
    if not raw.startswith("doc:"):
        return None

    body = raw[4:]                     # remove "doc:"

    # ------------------------------------------------------
    # doc:wiki:
    # ------------------------------------------------------

    if body.startswith("wiki:"):

        ref = body[len("wiki:"):]

        return (
            "doc:wiki",
            normalize_reference(ref)
        )

    # ------------------------------------------------------
    # doc:Temp_Import:
    # ------------------------------------------------------

    if body.startswith("Temp_Import:"):

        ref = body[len("Temp_Import:"):]

        return (
            "doc:Temp_Import",
            normalize_reference(ref)
        )

    # ------------------------------------------------------
    # doc:Temp_Import.
    # ------------------------------------------------------

    if body.startswith("Temp_Import."):

        return (
            "doc:Temp_Import",
            normalize_reference(body)
        )

    # ------------------------------------------------------
    # Normal doc
    # ------------------------------------------------------

    return (
        "doc",
        normalize_reference(body)
    )


# ==========================================================
# Extract ALL XML Links
# ==========================================================

all_links = {}

for row in df_clean.toLocalIterator():

    doc_id = row["doc_id"]
    reference = row["reference"]
    content = row["content"] or ""

    links = []

    # ------------------------------------------------------
    # Extract every wiki link
    # ------------------------------------------------------

    wiki_links = re.findall(r"\[\[(.*?)\]\]", content)

    for raw in wiki_links:

        parsed = parse_link(raw)

        if parsed is None:
            continue

        relation, ref = parsed

        if ref:

            links.append((relation, ref))

    # ------------------------------------------------------
    # Remove duplicates
    # ------------------------------------------------------

    links = sorted(set(links))

    all_links[doc_id] = links

    # ------------------------------------------------------
    # Display
    # ------------------------------------------------------

    print("=" * 80)
    print(f"Doc ID      : {doc_id}")
    print(f"Reference   : {reference}")
    print(f"Total Links : {len(links)}")
    print("-" * 80)

    for relation, ref in links:

        print(f"{relation:<18} --> {ref}")

    print()

# ==========================================================
# Statistics
# ==========================================================

relation_counter = {}

total_links = 0

for links in all_links.values():

    total_links += len(links)

    for relation, ref in links:

        relation_counter[relation] = relation_counter.get(relation, 0) + 1


print("=" * 80)
print("LINK EXTRACTION SUMMARY")
print("=" * 80)

print(f"Documents Processed : {len(all_links)}")
print(f"Total Links         : {total_links}")

print()

for relation in sorted(relation_counter):

    print(f"{relation:<20} : {relation_counter[relation]}")

print("=" * 80)

In [ ]:
# ==========================================================
# Cell 8 : Convert Links to RDF Triples
# ==========================================================

def resolve_reference(reference):
    """
    Resolve a reference to its document ID using the
    aliases created in Cell 5.
    """

    if not reference:
        return None

    reference = reference.strip()

    candidates = []

    # Original
    candidates.append(reference)

    # Remove anchor
    candidates.append(reference.split("||")[0])

    # Remove section
    candidates.append(reference.split("#")[0])

    # Replace ':' with '.'
    candidates.append(reference.replace(":", "."))

    # Remove duplicate dots
    candidates.extend([c.replace("..", ".") for c in candidates])

    # Remove trailing slash
    candidates.extend([c.rstrip("/") for c in candidates])

    # Remove spaces
    candidates.extend([c.strip() for c in candidates])

    # Remove duplicates while preserving order
    seen = set()
    final_candidates = []

    for c in candidates:
        if c and c not in seen:
            seen.add(c)
            final_candidates.append(c)

    # Lookup
    for c in final_candidates:
        if c in reference_to_docid:
            return reference_to_docid[c]

    return None


# ==========================================================
# Build RDF Triples
# ==========================================================

triples = set()

matched = 0
missing = 0
self_loops = 0

missing_references = []

relation_summary = {}

for parent_doc_id, links in all_links.items():

    for relation, reference in links:

        child_doc_id = resolve_reference(reference)

        if child_doc_id is None:

            missing += 1

            missing_references.append(
                (parent_doc_id, relation, reference)
            )

            continue

        matched += 1

        relation_summary[relation] = relation_summary.get(relation, 0) + 1

        if parent_doc_id == child_doc_id:

            self_loops += 1
            continue

        triples.add(
            (
                parent_doc_id,
                relation,
                child_doc_id
            )
        )


# ==========================================================
# Summary
# ==========================================================

print("=" * 80)
print("RDF TRIPLE SUMMARY")
print("=" * 80)

print(f"Matched References : {matched}")
print(f"Missing References : {missing}")
print(f"Self Loops         : {self_loops}")
print(f"Unique RDF Triples : {len(triples)}")

if matched + missing > 0:
    print(f"Match Rate         : {(matched/(matched+missing))*100:.2f}%")

print("=" * 80)

print("\nRelation Summary\n")

for relation, count in sorted(relation_summary.items()):
    print(f"{relation:<20} : {count}")

print("=" * 80)

print("\nSample RDF Triples\n")

triples = sorted(list(triples))

for triple in triples[:10]:
    print(triple)

# ==========================================================
# Missing References
# ==========================================================

if missing_references:

    print("\nFirst 20 Missing References\n")
    print("=" * 80)

    for parent, relation, ref in missing_references[:20]:

        print(f"Parent Doc : {parent}")
        print(f"Relation   : {relation}")
        print(f"Reference  : {ref}")
        print("-" * 80)

In [ ]:
# ==========================================================
# Cell 9 : Build NetworkX Knowledge Graph
# ==========================================================

import networkx as nx
from collections import Counter

# ==========================================================
# Create Directed Knowledge Graph
# ==========================================================

G = nx.DiGraph()

# ==========================================================
# Add Document Nodes
# ==========================================================

print("=" * 80)
print("ADDING DOCUMENT NODES")
print("=" * 80)

node_count = 0

for row in df_clean.toLocalIterator():

    G.add_node(
        row["doc_id"],

        title=row["title"],
        reference=row["reference"],
        version=row["version"],
        author=row["content_author"],
        update_date=row["content_update_date"],
        source_path=row["source_path"],
        content=row["content"]
    )

    node_count += 1

print(f"Nodes Added : {node_count}")

# ==========================================================
# Add RDF Relationships
# ==========================================================

print("\n" + "=" * 80)
print("ADDING RDF EDGES")
print("=" * 80)

edge_count = 0
relation_counter = Counter()

invalid_edges = []

for subject, relation, obj in triples:

    if subject not in G:
        invalid_edges.append((subject, relation, obj))
        continue

    if obj not in G:
        invalid_edges.append((subject, relation, obj))
        continue

    G.add_edge(
        subject,
        obj,
        relation=relation
    )

    relation_counter[relation] += 1
    edge_count += 1

print(f"Edges Added   : {edge_count}")
print(f"Invalid Edges : {len(invalid_edges)}")

# ==========================================================
# Graph Statistics
# ==========================================================

isolated_nodes = list(nx.isolates(G))

connected_nodes = G.number_of_nodes() - len(isolated_nodes)

density = nx.density(G)

avg_degree = 0

if G.number_of_nodes() > 0:
    avg_degree = (
        sum(dict(G.degree()).values())
        / G.number_of_nodes()
    )

print("\n")
print("=" * 80)
print("KNOWLEDGE GRAPH SUMMARY")
print("=" * 80)

print(f"Total Nodes           : {G.number_of_nodes()}")
print(f"Total Edges           : {G.number_of_edges()}")
print(f"Connected Nodes       : {connected_nodes}")
print(f"Isolated Nodes        : {len(isolated_nodes)}")
print(f"Average Degree        : {avg_degree:.2f}")
print(f"Graph Density         : {density:.6f}")

print("=" * 80)

# ==========================================================
# Relation Summary
# ==========================================================

print("\nRELATION DISTRIBUTION\n")

total_edges = max(1, G.number_of_edges())

for relation in sorted(relation_counter):

    count = relation_counter[relation]

    percent = (count / total_edges) * 100

    print(
        f"{relation:<20}"
        f"{count:>8}"
        f" ({percent:6.2f}%)"
    )

print("=" * 80)

# ==========================================================
# Top Connected Documents
# ==========================================================

print("\nTOP CONNECTED DOCUMENTS\n")

degree_list = sorted(
    G.degree(),
    key=lambda x: x[1],
    reverse=True
)

for node_id, degree in degree_list[:10]:

    data = G.nodes[node_id]

    print(
        f"Doc {node_id:<5}"
        f"Degree={degree:<4}"
        f"Title={data['title']}"
    )

print("=" * 80)

# ==========================================================
# Sample Nodes
# ==========================================================

print("\nSAMPLE NODES\n")

for i, (node_id, data) in enumerate(G.nodes(data=True)):

    print("-" * 80)

    print(f"Node ID      : {node_id}")
    print(f"Title        : {data['title']}")
    print(f"Reference    : {data['reference']}")
    print(f"Version      : {data['version']}")
    print(f"Author       : {data['author']}")

    if i == 4:
        break

# ==========================================================
# Sample Edges
# ==========================================================

print("\nSAMPLE EDGES\n")

for i, (src, dst, data) in enumerate(G.edges(data=True)):

    print(
        f"{src} "
        f"----[{data['relation']}]----> "
        f"{dst}"
    )

    if i == 9:
        break

# ==========================================================
# Invalid RDF Triples
# ==========================================================

if invalid_edges:

    print("\n")
    print("=" * 80)
    print("INVALID RDF TRIPLES (First 10)")
    print("=" * 80)

    for subject, relation, obj in invalid_edges[:10]:

        print(
            f"{subject}"
            f" --{relation}--> "
            f"{obj}"
        )

# ==========================================================
# Graph Validation
# ==========================================================

print("\n")
print("=" * 80)
print("GRAPH VALIDATION")
print("=" * 80)

print(f"Graph Type            : {type(G).__name__}")
print(f"Directed Graph        : {G.is_directed()}")
print(f"Self Loops            : {nx.number_of_selfloops(G)}")
print(f"Number of Components  : {nx.number_weakly_connected_components(G)}")

if len(invalid_edges) == 0:
    print("Graph Status          : PASS")
else:
    print("Graph Status          : WARNING")

print("=" * 80)

In [ ]:
# ==========================================================
# Cell 9.1 : Graph Analytics & Node Ranking
# ==========================================================

import networkx as nx
import pandas as pd

print("=" * 80)
print("GRAPH ANALYTICS")
print("=" * 80)

# ==========================================================
# Graph Metrics
# ==========================================================

pagerank_scores = nx.pagerank(G)

degree_scores = dict(G.degree())

in_degree_scores = dict(G.in_degree())

out_degree_scores = dict(G.out_degree())

betweenness_scores = nx.betweenness_centrality(G)

closeness_scores = nx.closeness_centrality(G)

# ==========================================================
# Build Node Analytics Index
# ==========================================================

node_index = []

for node in G.nodes():

    node_data = G.nodes[node]

    title = node_data.get("title", "")
    content = node_data.get("content", "")
    reference = node_data.get("reference", "")

    search_text = f"{title}\n{content}"

    analytics = {

        "doc_id": node,

        "title": title,

        "reference": reference,

        "pagerank": pagerank_scores.get(node, 0),

        "degree": degree_scores.get(node, 0),

        "in_degree": in_degree_scores.get(node, 0),

        "out_degree": out_degree_scores.get(node, 0),

        "betweenness": betweenness_scores.get(node, 0),

        "closeness": closeness_scores.get(node, 0),

        "search_text": search_text

    }

    node_index.append(analytics)

    # Store inside graph

    G.nodes[node]["pagerank"] = analytics["pagerank"]

    G.nodes[node]["degree"] = analytics["degree"]

    G.nodes[node]["in_degree"] = analytics["in_degree"]

    G.nodes[node]["out_degree"] = analytics["out_degree"]

    G.nodes[node]["betweenness"] = analytics["betweenness"]

    G.nodes[node]["closeness"] = analytics["closeness"]

    G.nodes[node]["search_text"] = search_text

# ==========================================================
# Convert to DataFrame
# ==========================================================

graph_index = pd.DataFrame(node_index)

graph_index = graph_index.sort_values(
    by="pagerank",
    ascending=False
).reset_index(drop=True)

# ==========================================================
# Summary
# ==========================================================

print("\n")
print("=" * 80)
print("GRAPH ANALYTICS SUMMARY")
print("=" * 80)

print(f"Total Nodes           : {G.number_of_nodes()}")

print(f"Total Edges           : {G.number_of_edges()}")

print(f"Average Degree        : {graph_index['degree'].mean():.2f}")

print(f"Maximum Degree        : {graph_index['degree'].max()}")

print(f"Maximum PageRank      : {graph_index['pagerank'].max():.6f}")

print(f"Minimum PageRank      : {graph_index['pagerank'].min():.6f}")

print("=" * 80)

# ==========================================================
# Top Important Documents
# ==========================================================

print("\nTOP 10 IMPORTANT DOCUMENTS\n")

for _, row in graph_index.head(10).iterrows():

    print("-" * 80)

    print(f"Doc ID         : {row.doc_id}")

    print(f"Title          : {row.title}")

    print(f"Reference      : {row.reference}")

    print(f"PageRank       : {row.pagerank:.6f}")

    print(f"Degree         : {row.degree}")

    print(f"In Degree      : {row.in_degree}")

    print(f"Out Degree     : {row.out_degree}")

    print(f"Betweenness    : {row.betweenness:.6f}")

    print(f"Closeness      : {row.closeness:.6f}")

print("-" * 80)

# ==========================================================
# Verify Graph Index
# ==========================================================

print("\nGRAPH INDEX SAMPLE\n")

print(
    graph_index[
        [
            "doc_id",
            "title",
            "pagerank",
            "degree",
            "in_degree",
            "out_degree"
        ]
    ].head(10)
)

print("\nGraph Analytics Completed Successfully.")

In [ ]:
# ==========================================================
# Cell 10 : Knowledge Graph Explorer
# ==========================================================

import matplotlib.pyplot as plt
import networkx as nx

# ==========================================================
# Number of Nodes to Visualize
# ==========================================================

TOP_NODES = 50

# ==========================================================
# Select Top PageRank Nodes
# ==========================================================

top_nodes = (
    graph_index
    .sort_values("pagerank", ascending=False)
    .head(TOP_NODES)["doc_id"]
    .tolist()
)

subgraph = G.subgraph(top_nodes).copy()

print("=" * 80)
print("KNOWLEDGE GRAPH EXPLORER")
print("=" * 80)

print(f"Original Nodes      : {G.number_of_nodes()}")
print(f"Original Edges      : {G.number_of_edges()}")

print(f"Displayed Nodes     : {subgraph.number_of_nodes()}")
print(f"Displayed Edges     : {subgraph.number_of_edges()}")

print("=" * 80)

# ==========================================================
# Layout
# ==========================================================

plt.figure(figsize=(20,15))

pos = nx.kamada_kawai_layout(subgraph)

# ==========================================================
# Node Sizes (PageRank)
# ==========================================================

node_sizes = []

for node in subgraph.nodes():

    pr = subgraph.nodes[node]["pagerank"]

    node_sizes.append(1000 + pr * 60000)

# ==========================================================
# Node Colors
# ==========================================================

node_colors = []

for node in subgraph.nodes():

    degree = subgraph.nodes[node]["degree"]

    if degree >= 10:
        node_colors.append("tomato")

    elif degree >= 5:
        node_colors.append("gold")

    else:
        node_colors.append("skyblue")

# ==========================================================
# Edge Colors
# ==========================================================

edge_colors = []

for u, v, data in subgraph.edges(data=True):

    relation = data["relation"]

    if relation == "doc":
        edge_colors.append("black")

    elif relation == "doc:wiki":
        edge_colors.append("blue")

    elif relation == "doc:Temp_Import":
        edge_colors.append("green")

    else:
        edge_colors.append("gray")

# ==========================================================
# Draw Nodes
# ==========================================================

nx.draw_networkx_nodes(

    subgraph,

    pos,

    node_size=node_sizes,

    node_color=node_colors,

    edgecolors="black",

    linewidths=2

)

# ==========================================================
# Draw Edges
# ==========================================================

nx.draw_networkx_edges(

    subgraph,

    pos,

    edge_color=edge_colors,

    arrows=True,

    arrowsize=20,

    width=2,

    alpha=0.8

)

# ==========================================================
# Labels
# ==========================================================

labels = {}

for node in subgraph.nodes():

    title = subgraph.nodes[node]["title"]

    if len(title) > 18:

        title = title[:18] + "..."

    labels[node] = f"{node}\n{title}"

nx.draw_networkx_labels(

    subgraph,

    pos,

    labels,

    font_size=8,

    font_weight="bold"

)

# ==========================================================
# Edge Labels
# ==========================================================

edge_labels = {}

for u,v,data in subgraph.edges(data=True):

    edge_labels[(u,v)] = data["relation"]

nx.draw_networkx_edge_labels(

    subgraph,

    pos,

    edge_labels=edge_labels,

    font_size=7

)

# ==========================================================
# Title
# ==========================================================

plt.title(

    "Knowledge Graph (Top PageRank Documents)",

    fontsize=18,

    fontweight="bold"

)

plt.axis("off")

plt.tight_layout()

plt.show()

# ==========================================================
# Top Important Documents
# ==========================================================

print("\nTOP IMPORTANT DOCUMENTS\n")

for _, row in graph_index.head(10).iterrows():

    print("-"*70)

    print(f"Doc ID      : {row.doc_id}")

    print(f"Title       : {row.title}")

    print(f"Reference   : {row.reference}")

    print(f"PageRank    : {row.pagerank:.6f}")

    print(f"Degree      : {row.degree}")

print("-"*70)

In [ ]:
# ==========================================================
# Cell 10 : Knowledge Graph Visualization
# ==========================================================

import matplotlib.pyplot as plt
import networkx as nx
from matplotlib.lines import Line2D

# ==========================================================
# Select Top Important Nodes
# ==========================================================

TOP_NODES = 50

top_nodes = (
    graph_index
    .sort_values("pagerank", ascending=False)
    .head(TOP_NODES)["doc_id"]
    .tolist()
)

subgraph = G.subgraph(top_nodes).copy()

print("=" * 80)
print("KNOWLEDGE GRAPH VISUALIZATION")
print("=" * 80)

print(f"Original Nodes  : {G.number_of_nodes()}")
print(f"Original Edges  : {G.number_of_edges()}")

print(f"Shown Nodes     : {subgraph.number_of_nodes()}")
print(f"Shown Edges     : {subgraph.number_of_edges()}")

print("=" * 80)

# ==========================================================
# Graph Layout
# ==========================================================

pos = nx.spring_layout(
    subgraph,
    seed=42,
    k=2,
    iterations=300
)

plt.figure(figsize=(22,16))

# ==========================================================
# Node Size (PageRank)
# ==========================================================

node_sizes = []

for node in subgraph.nodes():

    pr = subgraph.nodes[node]["pagerank"]

    node_sizes.append(2500 + pr * 100000)

# ==========================================================
# Node Color (Degree)
# ==========================================================

node_colors = []

for node in subgraph.nodes():

    degree = subgraph.nodes[node]["degree"]

    if degree >= 10:
        node_colors.append("#ff6b6b")

    elif degree >= 5:
        node_colors.append("#ffd93d")

    else:
        node_colors.append("#87ceeb")

# ==========================================================
# Edge Colors
# ==========================================================

edge_colors = []

for _, _, data in subgraph.edges(data=True):

    relation = data["relation"]

    if relation == "doc":
        edge_colors.append("black")

    elif relation == "doc:wiki":
        edge_colors.append("blue")

    elif relation == "doc:Temp_Import":
        edge_colors.append("green")

    else:
        edge_colors.append("gray")

# ==========================================================
# Draw Nodes
# ==========================================================

nx.draw_networkx_nodes(

    subgraph,

    pos,

    node_size=node_sizes,

    node_color=node_colors,

    edgecolors="black",

    linewidths=2

)

# ==========================================================
# Draw Directed Edges
# ==========================================================

nx.draw_networkx_edges(

    subgraph,

    pos,

    arrows=True,

    arrowstyle="-|>",

    arrowsize=30,

    width=2.5,

    edge_color=edge_colors,

    alpha=0.9,

    connectionstyle="arc3,rad=0.08",

    min_source_margin=20,

    min_target_margin=20

)

# ==========================================================
# Node Labels
# ==========================================================

labels = {}

for node in subgraph.nodes():

    title = subgraph.nodes[node]["title"]

    if len(title) > 18:
        title = title[:18] + "..."

    labels[node] = f"{node}\n{title}"

nx.draw_networkx_labels(

    subgraph,

    pos,

    labels,

    font_size=8,

    font_weight="bold"

)

# ==========================================================
# Edge Labels
# ==========================================================

edge_labels = {}

for u, v, data in subgraph.edges(data=True):

    edge_labels[(u, v)] = data["relation"]

nx.draw_networkx_edge_labels(

    subgraph,

    pos,

    edge_labels=edge_labels,

    font_size=8,

    rotate=False,

    label_pos=0.5,

    bbox=dict(

        facecolor="white",

        edgecolor="gray",

        alpha=0.85,

        boxstyle="round,pad=0.2"

    )

)

# ==========================================================
# Legend
# ==========================================================

legend = [

    Line2D(
        [0],[0],
        marker='o',
        color='w',
        label='Normal Document',
        markerfacecolor='#87ceeb',
        markersize=12
    ),

    Line2D(
        [0],[0],
        marker='o',
        color='w',
        label='Highly Connected',
        markerfacecolor='#ffd93d',
        markersize=12
    ),

    Line2D(
        [0],[0],
        marker='o',
        color='w',
        label='Hub Document',
        markerfacecolor='#ff6b6b',
        markersize=12
    ),

    Line2D(
        [0],[0],
        color='black',
        lw=2,
        label='doc'
    ),

    Line2D(
        [0],[0],
        color='blue',
        lw=2,
        label='doc:wiki'
    ),

    Line2D(
        [0],[0],
        color='green',
        lw=2,
        label='doc:Temp_Import'
    )

]

plt.legend(

    handles=legend,

    loc="upper left",

    fontsize=10

)

# ==========================================================
# Title
# ==========================================================

plt.title(

    "Knowledge Graph (Document Relationships)",

    fontsize=20,

    fontweight="bold"

)

plt.figtext(

    0.02,

    0.02,

    "Arrow Direction :  A  --->  B  means Document A refers to Document B",

    fontsize=11,

    color="darkred",

    fontweight="bold"

)

plt.axis("off")

plt.tight_layout()

plt.show()

# ==========================================================
# Top Important Documents
# ==========================================================

print("\nTOP IMPORTANT DOCUMENTS\n")

for _, row in graph_index.head(10).iterrows():

    print("-"*80)

    print(f"Doc ID      : {row.doc_id}")

    print(f"Title       : {row.title}")

    print(f"Reference   : {row.reference}")

    print(f"PageRank    : {row.pagerank:.6f}")

    print(f"Degree      : {row.degree}")

print("-"*80)

In [ ]:
# ==========================================================
# Cell 11A : Query Processing & Candidate Retrieval (Updated)
# ==========================================================

import re

STOP_WORDS = {
    "what","is","are","the","a","an","of","for",
    "to","in","on","about","how","does","do",
    "tell","me","explain","describe","show"
}

def normalize_query(query):
    words = re.findall(r"[a-zA-Z0-9_]+", query.lower())
    return [w for w in words if w not in STOP_WORDS]

def retrieve_candidates(graph_index, query, top_k=20):

    query_words = normalize_query(query)

    print("="*80)
    print("QUERY")
    print("="*80)
    print(query)
    print("\nNormalized Query")
    print(query_words)

    candidates=[]

    for _, row in graph_index.iterrows():

        title = str(row.get("title","")).lower()
        reference = str(row.get("reference","")).lower()
        searchable = str(row.get("search_text","")).lower()

        score = row.get("pagerank",0)*100 + row.get("degree",0)*2

        exact = 0
        partial = 0

        for q in query_words:

            if q in searchable:
                exact += 1
                score += 12

            elif len(q) >= 5:
                stem = q[:5]
                if stem in searchable:
                    partial += 1
                    score += 8

            if q in title:
                score += 20

            if q in reference:
                score += 15

        if exact==0 and partial==0:
            continue

        candidates.append({
            "doc_id":row["doc_id"],
            "title":row["title"],
            "reference":row["reference"],
            "pagerank":row["pagerank"],
            "degree":row["degree"],
            "score":score
        })

    if len(candidates)==0:
        print("\nNo exact matches found.")
        print("Falling back to highest ranked documents...\n")

        temp = graph_index.sort_values(
            by=["pagerank","degree"],
            ascending=False
        ).head(top_k)

        for _, row in temp.iterrows():
            candidates.append({
                "doc_id":row["doc_id"],
                "title":row["title"],
                "reference":row["reference"],
                "pagerank":row["pagerank"],
                "degree":row["degree"],
                "score":row["pagerank"]*100 + row["degree"]*2
            })

    candidates = sorted(
        candidates,
        key=lambda x:x["score"],
        reverse=True
    )[:top_k]

    print("\n"+"="*80)
    print("TOP CANDIDATES")
    print("="*80)

    for c in candidates:
        print(
            f'Doc:{c["doc_id"]:4}'
            f'  Score:{c["score"]:8.2f}'
            f'  PR:{c["pagerank"]:.5f}'
            f'  Deg:{c["degree"]:3}'
            f'  {c["title"]}'
        )

    return candidates


In [ ]:
# ==========================================================
# Cell 11B : Priority Knowledge Graph Traversal
# ==========================================================

from collections import deque

# ----------------------------------------------------------
# Graph Traversal
# ----------------------------------------------------------

def traverse_graph(

    graph,

    candidates,

    max_hops=2,

    max_nodes=50

):

    print("="*80)
    print("GRAPH TRAVERSAL")
    print("="*80)

    visited = set()

    queue = deque()

    retrieved = {}

    # ------------------------------------------------------
    # Initialize Queue
    # ------------------------------------------------------

    for item in candidates:

        queue.append(

            (

                item["doc_id"],

                0,

                None,

                "ROOT"

            )

        )

    # ------------------------------------------------------
    # Breadth First Traversal
    # ------------------------------------------------------

    while queue:

        current, hop, parent, relation = queue.popleft()

        if current in visited:
            continue

        if hop > max_hops:
            continue

        visited.add(current)

        node = graph.nodes[current]

        pagerank = node["pagerank"]

        degree = node["degree"]

        traversal_score = (

            pagerank * 100 +

            degree * 2 -

            hop * 5

        )

        retrieved[current] = {

            "doc_id": current,

            "title": node["title"],

            "reference": node["reference"],

            "pagerank": pagerank,

            "degree": degree,

            "hop": hop,

            "parent": parent,

            "relation": relation,

            "score": traversal_score

        }

        # --------------------------------------------------
        # Outgoing References
        # --------------------------------------------------

        outgoing = []

        for neighbor in graph.successors(current):

            edge = graph[current][neighbor]

            outgoing.append(

                (

                    neighbor,

                    edge["relation"],

                    graph.nodes[neighbor]["pagerank"]

                )

            )

        outgoing.sort(

            key=lambda x:x[2],

            reverse=True

        )

        for neighbor, rel, _ in outgoing:

            if neighbor not in visited:

                queue.append(

                    (

                        neighbor,

                        hop+1,

                        current,

                        rel

                    )

                )

        # --------------------------------------------------
        # Incoming References
        # --------------------------------------------------

        incoming = []

        for neighbor in graph.predecessors(current):

            edge = graph[neighbor][current]

            incoming.append(

                (

                    neighbor,

                    edge["relation"],

                    graph.nodes[neighbor]["pagerank"]

                )

            )

        incoming.sort(

            key=lambda x:x[2],

            reverse=True

        )

        for neighbor, rel, _ in incoming:

            if neighbor not in visited:

                queue.append(

                    (

                        neighbor,

                        hop+1,

                        current,

                        rel

                    )

                )

        if len(retrieved) >= max_nodes:

            break

    # ------------------------------------------------------
    # Final Ranking
    # ------------------------------------------------------

    ranked = sorted(

        retrieved.values(),

        key=lambda x:x["score"],

        reverse=True

    )

    print()

    print("="*80)

    print("RETRIEVED SUBGRAPH")

    print("="*80)

    for node in ranked:

        print(

            f'Doc:{node["doc_id"]:4} '

            f'Hop:{node["hop"]} '

            f'Score:{node["score"]:8.2f} '

            f'Parent:{node["parent"]} '

            f'Relation:{node["relation"]:<18} '

            f'{node["title"]}'

        )

    return ranked

In [ ]:
# ==========================================================
# Cell 11C : Build Knowledge Graph Package
# ==========================================================

def build_graph_package(
    query,
    graph,
    retrieved_nodes
):

    print("=" * 80)
    print("BUILDING KNOWLEDGE GRAPH PACKAGE")
    print("=" * 80)

    # ------------------------------------------------------
    # Documents
    # ------------------------------------------------------

    documents = []
    document_ids = set()

    for item in retrieved_nodes:

        node_id = item["doc_id"]

        if node_id in document_ids:
            continue

        document_ids.add(node_id)

        node = graph.nodes[node_id]

        documents.append({

            "doc_id": node_id,

            "title": node.get("title", ""),

            "reference": node.get("reference", ""),

            "content": node.get("content", ""),

            "search_text": node.get("search_text", ""),

            "pagerank": item["pagerank"],

            "degree": item["degree"],

            "hop": item["hop"],

            "parent": item["parent"],

            "relation": item["relation"],

            "score": item["score"]

        })

    # ------------------------------------------------------
    # Sort Documents
    # ------------------------------------------------------

    documents = sorted(
        documents,
        key=lambda x: x["score"],
        reverse=True
    )

    # ------------------------------------------------------
    # Graph Relationships
    # ------------------------------------------------------

    relationships = []

    for u, v, data in graph.edges(data=True):

        if u in document_ids and v in document_ids:

            relationships.append({

                "source_id": u,

                "source_title": graph.nodes[u].get("title", ""),

                "target_id": v,

                "target_title": graph.nodes[v].get("title", ""),

                "relation": data.get("relation", "")

            })

    # ------------------------------------------------------
    # Sort Relationships
    # ------------------------------------------------------

    relationships = sorted(

        relationships,

        key=lambda x: (
            x["source_id"],
            x["target_id"],
            x["relation"]
        )

    )

    # ------------------------------------------------------
    # Graph Package
    # ------------------------------------------------------

    graph_package = {

        "query": query,

        "documents": documents,

        "relationships": relationships,

        "statistics": {

            "documents": len(documents),

            "relationships": len(relationships)

        }

    }

    # ------------------------------------------------------
    # Summary
    # ------------------------------------------------------

    print()
    print("=" * 80)
    print("GRAPH PACKAGE SUMMARY")
    print("=" * 80)

    print(f"Query               : {query}")
    print(f"Retrieved Documents : {len(documents)}")
    print(f"Relationships       : {len(relationships)}")

    print()
    print("=" * 80)
    print("DOCUMENTS")
    print("=" * 80)

    for doc in documents:

        print(

            f'Doc:{doc["doc_id"]:4} '

            f'Hop:{doc["hop"]} '

            f'Score:{doc["score"]:8.2f} '

            f'{doc["title"]}'

        )

    print()
    print("=" * 80)
    print("RELATIONSHIPS")
    print("=" * 80)

    for rel in relationships:

        print(

            f'{rel["source_id"]} '

            f'({rel["source_title"]}) '

            f'---{rel["relation"]}---> '

            f'{rel["target_id"]} '

            f'({rel["target_title"]})'

        )

    print()
    print("=" * 80)
    print("GRAPH PACKAGE READY")
    print("=" * 80)

    return graph_package

In [ ]:
# ==========================================================
# Cell 12 : GraphRAG Pipeline
# Part 1 : GraphRAG Class Initialization
# ==========================================================

import networkx as nx


class GraphRAG:

    # ------------------------------------------------------
    # Constructor
    # ------------------------------------------------------

    def __init__(
        self,
        graph,
        graph_index
    ):

        self.graph = graph

        self.graph_index = graph_index

    # ------------------------------------------------------
    # Retrieve Knowledge
    # ------------------------------------------------------

    def retrieve(

        self,

        query,

        top_k=10,

        max_hops=2,

        max_nodes=50

    ):

        print("=" * 80)
        print("GRAPHRAG RETRIEVAL PIPELINE")
        print("=" * 80)

        print(f"Query : {query}")

        print()

        # ==================================================
        # Step 1
        # Candidate Retrieval
        # (Cell 11A)
        # ==================================================
        candidates = retrieve_candidates(

            graph_index=self.graph_index,

            query=query,

            top_k=top_k

        )

        # --------------------------------------------------
        # No Candidate Found
        # --------------------------------------------------

        if len(candidates) == 0:

            print("No candidate documents found.")

            return {

                "query": query,

                "candidates": [],

                "retrieved_nodes": [],

                "retrieved_subgraph": nx.DiGraph(),

                "graph_package": None

            }

        # ==================================================
        # Step 2
        # Knowledge Graph Traversal
        # (Cell 11B)
        # ==================================================

        retrieved_nodes = traverse_graph(

            graph=self.graph,

            candidates=candidates,

            max_hops=max_hops,

            max_nodes=max_nodes

        )

        # ==================================================
        # Step 3
        # Build Knowledge Graph Package
        # (Cell 11C)
        # ==================================================

        graph_package = build_graph_package(

            query=query,

            graph=self.graph,

            retrieved_nodes=retrieved_nodes

        )

        # ==================================================
        # Step 4
        # Create Retrieved Subgraph
        # ==================================================

        retrieved_doc_ids = [

            node["doc_id"]

            for node in retrieved_nodes

        ]

        retrieved_subgraph = self.graph.subgraph(

            retrieved_doc_ids

        ).copy()

        print()

        print("=" * 80)
        print("RETRIEVED SUBGRAPH")
        print("=" * 80)

        print(f"Nodes : {retrieved_subgraph.number_of_nodes()}")

        print(f"Edges : {retrieved_subgraph.number_of_edges()}")

        print()
        # ==================================================
        # Step 5
        # Retrieval Summary
        # ==================================================

        print("=" * 80)
        print("RETRIEVAL SUMMARY")
        print("=" * 80)

        print(f"Candidate Documents : {len(candidates)}")

        print(f"Retrieved Nodes     : {len(retrieved_nodes)}")

        print(f"Subgraph Nodes      : {retrieved_subgraph.number_of_nodes()}")

        print(f"Subgraph Edges      : {retrieved_subgraph.number_of_edges()}")

        print()

        # ==================================================
        # Prepare Retrieval Result
        # ==================================================

        retrieval_result = {

            "query": query,

            "candidates": candidates,

            "retrieved_nodes": retrieved_nodes,

            "retrieved_subgraph": retrieved_subgraph,

            "graph_package": graph_package

        }

        print("=" * 80)
        print("GRAPHRAG RETRIEVAL COMPLETED")
        print("=" * 80)

        print()
        # ==================================================
        # Step 6
        # Return Retrieval Package
        # ==================================================

        return retrieval_result

cell 13 LLM

In [ ]:
# ==========================================================
# Cell 14 : GraphRAG Retrieval
# ==========================================================

# ----------------------------------------------------------
# User Query
# ----------------------------------------------------------

query = """
What phenomenon can occur when two vibrating structures come
into contact at matching wave velocities, and what design
features can reduce this risk on engine casings?
"""

# ----------------------------------------------------------
# Initialize GraphRAG
# ----------------------------------------------------------

rag = GraphRAG(

    graph=G,

    graph_index=graph_index

)

# ----------------------------------------------------------
# Run Retrieval Pipeline
# ----------------------------------------------------------

result = rag.retrieve(

    query=query,

    top_k=10,

    max_hops=2,

    max_nodes=50

)

# ----------------------------------------------------------
# Retrieval Summary
# ----------------------------------------------------------

print("\n")

print("=" * 90)
print("GRAPH RAG RETRIEVAL")
print("=" * 90)

print(f"\nUser Query :\n{result['query']}")

# ----------------------------------------------------------
# Retrieved Documents
# ----------------------------------------------------------

print("\n")

print("=" * 90)
print("RETRIEVED DOCUMENTS")
print("=" * 90)

for node in result["retrieved_nodes"]:

    print(f"\nDocument ID : {node['doc_id']}")

    print(f"Title       : {node['title']}")

    print(f"Reference   : {node['reference']}")

    print(f"PageRank    : {node['pagerank']:.6f}")

    print(f"Degree      : {node['degree']}")

    print(f"Hop         : {node['hop']}")

    print(f"Score       : {node['score']:.2f}")

# ----------------------------------------------------------
# Retrieved Relationships
# ----------------------------------------------------------

print("\n")

print("=" * 90)
print("RETRIEVED RELATIONSHIPS")
print("=" * 90)

retrieved_graph = result["retrieved_subgraph"]

for source, target, data in retrieved_graph.edges(data=True):

    relation = data.get("relation", "related_to")

    print(

        f"{source} ({G.nodes[source]['title']}) "

        f"---{relation}---> "

        f"{target} ({G.nodes[target]['title']})"

    )

# ----------------------------------------------------------
# Retrieval Statistics
# ----------------------------------------------------------

print("\n")

print("=" * 90)

print(f"Retrieved Nodes : {retrieved_graph.number_of_nodes()}")

print(f"Retrieved Edges : {retrieved_graph.number_of_edges()}")

print("=" * 90)

In [ ]:
# ==========================================================
# Cell 15 : Final Answer + Retrieved Subgraph
# ==========================================================

# ----------------------------------------------------------
# Build Prompt
# ----------------------------------------------------------

graph_package = result["graph_package"]

prompt = f"""
You are an expert technical assistant.

Answer ONLY using the information contained in the retrieved
Knowledge Graph.

If the answer is not present in the retrieved documents,
reply:

'I could not find the answer in the retrieved Knowledge Graph.'

==================================================
Question
==================================================

{result["query"]}

==================================================
Retrieved Documents
==================================================

{graph_package["documents"]}

==================================================
Relationships
==================================================

{graph_package["relationships"]}

Provide a detailed technical answer.
"""

# ----------------------------------------------------------
# Generate Answer
# ----------------------------------------------------------

response = llm.invoke(prompt)

# ----------------------------------------------------------
# Visualize Retrieved Subgraph
# ----------------------------------------------------------

visualize_graph(

    result["retrieved_subgraph"]

)

# ----------------------------------------------------------
# Final Output
# ----------------------------------------------------------

print("\n")

print("=" * 90)
print("QUESTION")
print("=" * 90)

print(result["query"])

print("\n")

print("=" * 90)
print("ANSWER")
print("=" * 90)

print(response.content)